In [1]:
from rlm import RLM
from rlm.core.config import DockerConfig, ParseConfig, WorkspaceConfig
from rlm.logger import RLMLogger

In [2]:
logger = RLMLogger(log_dir="./logs")

rlm = RLM(
    backend="vllm",
    backend_kwargs={
        "model_name": "qwen3-5-9b",
        "base_url": "http://127.0.0.1:8008/v1",
        "api_key": "EMPTY",
        "temperature": 0.7, 
        "top_p": 0.8, 
        "top_k": 20, 
        "presence_penalty": 1.5,
        "repetition_penalty": 1.0
    },
    workspace_config=WorkspaceConfig(
        parse=ParseConfig(action_format="native"),
        docker=DockerConfig(cleanup_mode="keep"),
    ),
    logger=logger,
    verbose=True,
)



╭─ ◆ RLM ━ Recursive Language Model ──────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    Backend                            vllm                    Environment                       docker          │
│    Model                              qwen3-5-9b              Action Format                     native          │
│    Max Iterations                     30                      Max Depth                         1               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
exercise_1_latex = (
    "\\textbf{Exercise 2} Assume $A$ is an $n\\times n$ SPD matrix, and let "
    "$\\{p_{j}\\}_{j=1}^{n}$ be a set of $A$-orthogonal vectors. \n\n"
    "A solution to $Ax=b$ is given by $\\overline{x}=\\sum_{j=1}^{n}\\alpha_{j}p_{j}$ "
    "where $\\alpha_{j}=\\frac{\\langle b,p_{j}\\rangle}{\\langle p_{j},Ap_{j}\\rangle}$. \n\n"
    "Show that among all approximations of the form $\\hat{x}=\\sum_{j=1}^{k}\\gamma_{j}p_{j}$ "
    "the one that minimizes $||\\overline{x}-\\hat{x}||_{A}$ is obtained by setting "
    "$\\gamma_{j}=\\alpha_{j}$. "
)

exercise_1_latex = (
    "\\textbf{Exercise 1} Using the facts proven in class about $A$-orthogonal vectors "
    "and about vectors in the Conjugate-Gradient method, show that the following "
    "identities hold for the coefficients occurring in the Conjugate-Gradient method:\n\n"
    "\\begin{equation}\n"
    "\\alpha_{k}=\\frac{\\langle b,p_{k}\\rangle}{\\langle p_{k},Ap_{k}\\rangle}="
    "\\frac{\\langle r_{k-1},p_{k}\\rangle}{\\langle p_{k},Ap_{k}\\rangle}="
    "\\frac{\\langle r_{k-1},r_{k-1}\\rangle}{\\langle p_{k},Ap_{k}\\rangle}\n"
    "\\end{equation}\n\n"
    "\\begin{equation}\n"
    "\\beta_{k}=-\\frac{\\langle r_{k-1},Ap_{k-1}\\rangle}{\\langle p_{k-1},Ap_{k-1}\\rangle}="
    "\\frac{\\langle r_{k-1},r_{k-1}\\rangle}{\\langle r_{k-2},r_{k-2}\\rangle}\n"
    "\\end{equation}\n\n"
    "By proving these identities, you are validating the formulas that are used in "
    "typical implementations of the Conjugate-Gradient method."
)

In [4]:
print(exercise_1_latex)

\textbf{Exercise 1} Using the facts proven in class about $A$-orthogonal vectors and about vectors in the Conjugate-Gradient method, show that the following identities hold for the coefficients occurring in the Conjugate-Gradient method:

\begin{equation}
\alpha_{k}=\frac{\langle b,p_{k}\rangle}{\langle p_{k},Ap_{k}\rangle}=\frac{\langle r_{k-1},p_{k}\rangle}{\langle p_{k},Ap_{k}\rangle}=\frac{\langle r_{k-1},r_{k-1}\rangle}{\langle p_{k},Ap_{k}\rangle}
\end{equation}

\begin{equation}
\beta_{k}=-\frac{\langle r_{k-1},Ap_{k-1}\rangle}{\langle p_{k-1},Ap_{k-1}\rangle}=\frac{\langle r_{k-1},r_{k-1}\rangle}{\langle r_{k-2},r_{k-2}\rangle}
\end{equation}

By proving these identities, you are validating the formulas that are used in typical implementations of the Conjugate-Gradient method.


In [5]:
result = rlm.completion(
    exercise_1_latex)

╭─ ★ Final Answer ────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Proof of Conjugate-Gradient Coefficient Identities                                                             │
│                                                                                                                 │
│  ## Preliminaries                                                                                               │
│                                                                                                                 │
│  The Conjugate-Gradient (CG) method solves Ax = b where A is symmetric positive definite.                       │
│                                                                                                                 │
│  **Initialization:**                                                                                            │
│  - x_0: initial guess                                                                                           │
│  - r_0 = b - Ax_0: initial residual                                                                             │
│  - p_0 = r_0: initial search direction                                                                          │
│                                                                                                                 │
│  **Iterative step for k = 1, 2, ...**:                                                                          │
│  - r_k = r_{k-1} - α_k A p_{k-1}                                                                                │
│  - p_k = r_k + β_k p_{k-1}                                                                                      │
│                                                                                                                 │
│  **Key properties:**                                                                                            │
│  1. Residual orthogonality: ⟨r_i, r_j⟩ = 0 for i ≠ j                                                            │
│  2. A-orthogonality of search directions: ⟨p_i, A p_j⟩ = 0 for i ≠ j                                            │
│  3. Optimality: r_k ⟂ K_k(A, r_0)                                                                               │
│  4. Therefore: r_k ⟂ r_{k-1} and p_k ⟂ A p_{k-1}                                                                │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Part 1: Proof for α_k                                                                                       │
│                                                                                                                 │
│  ### Step 1: Show α_k = ⟨r_{k-1}, r_{k-1}⟩ / ⟨A p_{k-1}, r_{k-1}⟩                                               │
│                                                                                                                 │
│  From r_k = r_{k-1} - α_k A p_{k-1} and r_k ⟂ r_{k-1}:                                                          │
│  ⟨r_k, r_{k-1}⟩ = 0                                                                                             │
│  ⟨r_{k-1} - α_k A p_{k-1}, r_{k-1}⟩ = 0                                                                         │
│  ||r_{k-1}||² - α_k ⟨A p_{k-1}, r_{k-1}⟩ = 0                                                                    │
│  α_k = ||r_{k-1}||² / ⟨A p_{k-1}, r_{k-1}⟩                                                                      │
│                                                       

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Iterations       26/30                                                
                                             Total Time       1014.06s                                             
                                             Input Tokens     292,558                                              
                                             Output Tokens    76,838    

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

In [7]:
print(result.response)

The identities for Conjugate-Gradient method coefficients are proven as follows:

**Identity 1: $\alpha_k = \frac{\langle r_{k-1},p_k \rangle}{\langle p_k,Ap_k \rangle} = \frac{\langle r_{k-1},r_{k-1} \rangle}{\langle p_k,Ap_k \rangle}$**

1. From the minimization principle: $\langle r_k, p_k \rangle = 0$ where $r_k = r_{k-1} - \alpha_k A p_k$
2. This gives: $\langle r_{k-1} - \alpha_k A p_k, p_k \rangle = 0 \implies \alpha_k = \frac{\langle r_{k-1}, p_k \rangle}{\langle p_k, A p_k \rangle}$
3. From CG Krylov subspace properties: $\langle r_{k-1}, p_k \rangle = \|r_{k-1}\|^2$

**Identity 2: $\beta_k = -\frac{\langle r_{k-1},Ap_{k-1}\rangle}{\langle p_{k-1},Ap_{k-1}\rangle} = \frac{\langle r_{k-1},r_{k-1} \rangle}{\langle r_{k-2},r_{k-2} \rangle}$**

1. From A-orthogonality and residual update: $\langle r_{k-1}, A p_{k-1} \rangle = -\beta_k \langle p_{k-1}, A p_{k-1} \rangle$
2. From residual orthogonality: $\beta_k = \frac{\|r_{k-1}\|^2}{\|r_{k-2}\|^2}$

Both identities validate the fo

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="EMPTY",
    base_url="http://127.0.0.1:8008/v1",
)

response = client.chat.completions.create(
    model="qwen3-5-9b",
    messages=[
        {"role": "user", "content": exercise_1_latex}
    ]
)

In [ ]:
print(response.choices[0].message.content)

In [ ]:
msg = response.choices[0].message
reasoning = getattr(msg, "reasoning_content", None)
if reasoning is None and hasattr(msg, "model_extra") and msg.model_extra:
    reasoning = msg.model_extra.get("reasoning_content")

if reasoning:
    print("=== REASONING ===\n")
    print(reasoning)
else:
    print("No explicit reasoning_content found. Printing full message for inspection:\n")
    print(msg)